# Feature Engineering
**Crop Classification System**

This notebook handles missing values, encodes categoricals, scales features, and creates derived features.

## Setup

In [8]:
import pandas as pd
import numpy as np
import pickle
import os
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder, StandardScaler

np.random.seed(42)

df = pd.read_csv('../data/dirty_dataset.csv')
print(f"Dirty dataset shape: {df.shape}")
print(f"Total missing values: {df.isnull().sum().sum()}")

Dirty dataset shape: (42949, 20)
Total missing values: 37751


## Remove Duplicates

In [9]:
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicate rows")
print(f"Shape after dedup: {df.shape}")

Removed 216 duplicate rows
Shape after dedup: (42733, 20)


## Outlier Treatment (IQR Method)

In [10]:
# Cap extreme outliers using 1st and 99th percentile
outlier_cols = ['Yield_kg_per_ha', 'Fertilizer_kg_per_ha', 'Rainfall_mm',
                'Irrigation_1000ha', 'N_kg_per_ha', 'P_kg_per_ha', 'K_kg_per_ha']

for col in outlier_cols:
    Q1 = df[col].quantile(0.01)
    Q3 = df[col].quantile(0.99)
    df[col] = df[col].clip(lower=Q1, upper=Q3)
    print(f"{col}: clipped to [{Q1:.2f}, {Q3:.2f}]")

print("\n Outliers capped via winsorization (preserving real-world variance)")

Yield_kg_per_ha: clipped to [308.23, 3775.16]
Fertilizer_kg_per_ha: clipped to [20.53, 253.77]
Rainfall_mm: clipped to [700.00, 1200.00]
Irrigation_1000ha: clipped to [-1.00, 56.30]
N_kg_per_ha: clipped to [7.03, 84.53]
P_kg_per_ha: clipped to [3.45, 38.98]
K_kg_per_ha: clipped to [5.95, 63.56]

 Outliers capped via winsorization (preserving real-world variance)


## Missing Value Imputation


In [11]:
median_cols = ['Rainfall_mm','Irrigation_1000ha','Fertilizer_kg_per_ha',
               'Humidity_pct','Soil_Moisture_pct','Pesticide_kg_per_ha',
               'Groundwater_Level_m','Mechanization_Score','pH','Wind_Speed_m_s']

for col in median_cols:
    missing_before = df[col].isnull().sum()
    group_med = df.groupby(['Crop','State'])[col].transform('median')
    global_med = df[col].median()
    df[col] = df[col].fillna(group_med).fillna(global_med)
    print(f"{col:<30}: filled {missing_before} NaNs ")

print(f"\nTotal missing after imputation: {df.isnull().sum().sum()}")

Rainfall_mm                   : filled 2989 NaNs 
Irrigation_1000ha             : filled 4272 NaNs 
Fertilizer_kg_per_ha          : filled 3844 NaNs 
Humidity_pct                  : filled 2989 NaNs 
Soil_Moisture_pct             : filled 5126 NaNs 
Pesticide_kg_per_ha           : filled 4699 NaNs 
Groundwater_Level_m           : filled 3417 NaNs 
Mechanization_Score           : filled 5554 NaNs 
pH                            : filled 2562 NaNs 
Wind_Speed_m_s                : filled 2135 NaNs 

Total missing after imputation: 0


## Derived Feature Engineering

In [12]:
# NPK Balance — total nutrient requirement proxy
df['NPK_Balance'] = df['N_kg_per_ha'] + df['P_kg_per_ha'] + df['K_kg_per_ha']

# Water Stress Index — rainfall relative to irrigation dependency
df['Water_Stress_Index'] = df['Rainfall_mm'] / (df['Irrigation_1000ha'].clip(lower=0.01) + 1)

# Fertilizer Efficiency — yield per unit fertilizer
df['Fert_Efficiency'] = df['Yield_kg_per_ha'] / (df['Fertilizer_kg_per_ha'].clip(lower=0.01) + 1)

# Temperature-Humidity Index (THI) — heat-moisture stress on crops
hum = df['Humidity_pct'].clip(0, 100) / 100
temp = df['Temperature_C']
df['THI'] = temp + 0.33 * (hum * 6.105 * np.exp(17.27 * temp / (237.7 + temp))) - 4

# Normalized year (0–1 temporal trend feature)
df['Year_norm'] = (df['Year'] - df['Year'].min()) / (df['Year'].max() - df['Year'].min())

# Guard against any inf/nan from derived calculations
for col in ['NPK_Balance','Water_Stress_Index','Fert_Efficiency','THI','Year_norm']:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(df[col].median())

print("Derived features created:")
print("  - NPK_Balance          (N + P + K total nutrient load)")
print("  - Water_Stress_Index   (Rainfall / Irrigation ratio)")
print("  - Fert_Efficiency      (Yield per kg fertilizer)")
print("  - THI                  (Temperature-Humidity Index)")
print("  - Year_norm            (Normalized year for temporal trends)")
print(f"\nNew shape: {df.shape}")

Derived features created:
  - NPK_Balance          (N + P + K total nutrient load)
  - Water_Stress_Index   (Rainfall / Irrigation ratio)
  - Fert_Efficiency      (Yield per kg fertilizer)
  - THI                  (Temperature-Humidity Index)
  - Year_norm            (Normalized year for temporal trends)

New shape: (42733, 25)


## Categorical Encoding

In [13]:
le_crop     = LabelEncoder()
le_state    = LabelEncoder()
le_district = LabelEncoder()

df['Crop_encoded']     = le_crop.fit_transform(df['Crop'])
df['State_encoded']    = le_state.fit_transform(df['State'])
df['District_encoded'] = le_district.fit_transform(df['District'])

print("Label Encoding complete:")
for i, cls in enumerate(le_crop.classes_):
    print(f"  {i} → {cls}")

Label Encoding complete:
  0 → Cotton
  1 → Groundnut
  2 → Maize
  3 → Pearl Millet
  4 → Rice
  5 → Sorghum
  6 → Sugarcane
  7 → Wheat


## Feature Scaling (StandardScaler)

In [14]:
feature_cols = [
    'Rainfall_mm','Irrigation_1000ha','Fertilizer_kg_per_ha',
    'Yield_kg_per_ha','Temperature_C','Humidity_pct',
    'Soil_Moisture_pct','Pesticide_kg_per_ha','Groundwater_Level_m',
    'Mechanization_Score','N_kg_per_ha','P_kg_per_ha','K_kg_per_ha',
    'pH','Altitude_m','Wind_Speed_m_s',
    'NPK_Balance','Water_Stress_Index','Fert_Efficiency','THI','Year_norm',
    'State_encoded','District_encoded'
]

# Final NaN safety check
for col in feature_cols:
    df[col] = df[col].fillna(df[col].median())

scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[feature_cols] = scaler.fit_transform(df[feature_cols].astype(float))

print(f"Features scaled: {len(feature_cols)}")
print("Mean ~0 after scaling:", df_scaled[feature_cols].mean().abs().max().round(10))

Features scaled: 23
Mean ~0 after scaling: 0.0


## Save Processed Dataset

In [15]:
os.makedirs('../models', exist_ok=True)
for name, obj in [
    ('label_encoder_crop',     le_crop),
    ('label_encoder_state',    le_state),
    ('label_encoder_district', le_district),
    ('scaler',                 scaler),
    ('feature_cols',           feature_cols),
]:
    with open(f'../models/{name}.pkl', 'wb') as f:
        pickle.dump(obj, f)

df_scaled.to_csv('../data/processed_dataset.csv', index=False)
print(f" processed_dataset.csv saved → {df_scaled.shape}")
print(f" Encoders & scaler saved to /models/")

 processed_dataset.csv saved → (42733, 28)
 Encoders & scaler saved to /models/


## Summary

| Step | Action | Result |
|------|--------|--------|
| Dedup | Drop exact duplicates | ~216 rows removed |
| Outliers | 1–99th percentile clipping | Extreme values capped |
| Imputation | Group-wise median (Crop+State) | 0 missing values |
| Derived | NPK_Balance, THI, Efficiency | 5 new features |
| Encoding | LabelEncoder (Crop, State, District) | Integer class codes |
| Scaling | StandardScaler | Zero mean, unit variance |
